# Chunking Experiments

Notebook for exploring different chunking strategies (`recursive`, `semantic`, `sentence`)
and their effect on retrieval quality.  Compare chunk size, overlap, and boundary heuristics
against a set of golden queries.


In [1]:
%load_ext autoreload
# TODO: %autoreload 2

# TODO: from src.ingestion.loader import PolicyDocumentLoader
# TODO: from src.ingestion.chunker import PolicyChunker
# TODO: import pandas as pd
# TODO: import matplotlib.pyplot as plt

In [2]:
# TODO: Load raw documents
# loader = PolicyDocumentLoader(source_dir='data/raw')
# documents = loader.load()
# print(f'Loaded {len(documents)} documents')

In [3]:
# TODO: Experiment 1 — vary chunk_size
# results = []
# for size in [256, 512, 1024]:
#     chunker = PolicyChunker(chunk_size=size, chunk_overlap=64)
#     chunks = chunker.split(documents)
#     results.append({'chunk_size': size, 'n_chunks': len(chunks)})
# pd.DataFrame(results)

In [4]:
# TODO: Visualise chunk length distribution
# lengths = [len(c.page_content) for c in chunks]
# plt.hist(lengths, bins=30)
# plt.xlabel('Chunk length (chars)')
# plt.ylabel('Count')
# plt.title('Chunk Length Distribution')
# plt.show()

In [2]:
import sys
sys.path.append("..")  # so we can import from src/

from src.ingestion.loader import PolicyDocumentLoader
from src.ingestion.chunker import PolicyChunker

In [6]:
loader = PolicyDocumentLoader("../data/raw")
docs = loader.load()

print(f"Pages loaded: {len(docs)}")
print(f"First page metadata: {docs[0].metadata}")
print(f"First page preview:\n{docs[0].page_content[:300]}")

Skipping unsupported file type: ../data/raw/.gitkeep
Pages loaded: 96
First page metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2021-09-01T11:56:41-04:00', 'author': 'U.S. Office of Personnel Management', 'keywords': 'pay; leave; eregency preparedness; weather;', 'moddate': '2021-09-01T13:20:22-04:00', 'subject': 'Handbook on Pay and Leave Benefits for Federal Employees Affected by Severe Weather Conditions or Other Emergency Situations', 'title': 'Handbook on Pay and Leave Benefits for Federal Employees Affected by Severe Weather Conditions or Other Emergency Situations', 'source': '../data/raw/emergencybenefits.pdf', 'total_pages': 20, 'page': 0, 'page_label': 'i'}
First page preview:
UNITED STATES OFFICE OF PERSONNEL MANAGEMENT 
Handbook on Pay and Leave Benefts 
for Federal Employees 
Afected by Severe Weather Conditions 
or Other Emergency Situations 
OPM.GOV SEPTEMBER 2021


In [7]:
chunker = PolicyChunker(chunk_size=512, chunk_overlap=64, strategy="recursive")
chunks = chunker.split(docs)

print(f"Total chunks: {len(chunks)}")
print(f"\nChunk 0 metadata: {chunks[0].metadata}")
print(f"Chunk 0 content:\n{chunks[0].page_content}")

Total chunks: 433

Chunk 0 metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2021-09-01T11:56:41-04:00', 'author': 'U.S. Office of Personnel Management', 'keywords': 'pay; leave; eregency preparedness; weather;', 'moddate': '2021-09-01T13:20:22-04:00', 'subject': 'Handbook on Pay and Leave Benefits for Federal Employees Affected by Severe Weather Conditions or Other Emergency Situations', 'title': 'Handbook on Pay and Leave Benefits for Federal Employees Affected by Severe Weather Conditions or Other Emergency Situations', 'source': '../data/raw/emergencybenefits.pdf', 'total_pages': 20, 'page': 2, 'page_label': '1', 'chunk_index': 3}
Chunk 0 content:
1 
I. Pay and Leave Benefits For Employees Prevented From Working In an Area
Affected By Severe Weather Conditions or Other Emergency Situations
Weather and Safety Leave 
Weather and safety leave is provided at an agency’s discretion for such events related to 
the

I noticed cover pages and TOC sections were being indexed so I added a minimum content length filter to drop chunks below X characters.

In [8]:
chunker_char = PolicyChunker(chunk_size=512, chunk_overlap=64, strategy="character")
chunks_char = chunker_char.split(docs)

print(f"Recursive chunks: {len(chunks)}")
print(f"Character chunks: {len(chunks_char)}")


Recursive chunks: 433
Character chunks: 89


Compared recursive character splitting vs fixed character splitting on 96 pages of OPM compliance documents. Character splitting produced 96 chunks — one per page — exceeding the 512 token target. Recursive splitting produced 478 chunks by respecting paragraph and sentence boundaries, staying within the token budget while preserving semantic coherence

In [9]:
# inspect a mid-document recursive chunk to check quality
for chunk in chunks[10:15]:
    print("---")
    print(f"source: {chunk.metadata['source']}")
    print(f"page: {chunk.metadata['page']}")
    print(f"chunk_index: {chunk.metadata['chunk_index']}")
    print(f"length: {len(chunk.page_content)} chars")
    print(f"content:\n{chunk.page_content}")
    print()

---
source: ../data/raw/emergencybenefits.pdf
page: 3
chunk_index: 13
length: 507 chars
content:
considered to be regularly entitled to night pay differential and Sunday premium pay for 
applicable hours in the employee’s normal basic workweek.  Also, an employee is 
considered to be regularly entitled to law enforcement availability pay, administratively 
uncontrollable overtime pay, standby duty premium pay, regular overtime pay for 
firefighters covered by 5 U.S.C. 5545b, retention payments under 5 U.S.C. 5754, 
physicians’ comparability allowances, and supervisory differentials, as applicable.

---
source: ../data/raw/emergencybenefits.pdf
page: 3
chunk_index: 14
length: 424 chars
content:
Agencies must make all deductions from advance payments that are authorized by law, 
including retirement or Social Security (FICA) deductions, authorized allotments, and 
income tax withholdings.  An employee’s receipt of an advance payment may not affect 
the amount of the evacuation payments o

chunk 10 — it ends with "The agency must determine the days and hours the employee would have been expected to" and chunk 11 picks up with "work during the selected time period". That's a mid-sentence cut across a chunk boundary.
This is exactly what chunk_overlap=64 is supposed to help with but the overlap isn't showing here between chunk 10 and 11. The sentence is still broken.
This tells us 512 characters might be slightly small for dense government policy text where sentences run long. 

In [10]:
chunker_larger = PolicyChunker(chunk_size=1024, chunk_overlap=128, strategy="recursive")
chunks_larger = chunker_larger.split(docs)

print(f"512/64 chunks: {len(chunks)}")
print(f"1024/128 chunks: {len(chunks_larger)}")

# check same area
for chunk in chunks_larger[5:8]:
    print("---")
    print(f"length: {len(chunk.page_content)} chars")
    print(f"content:\n{chunk.page_content}")
    print()

512/64 chunks: 433
1024/128 chunks: 250
---
length: 1008 chars
content:
biweekly pay period).  If possible, the agency should estimate an intermittent employee’s 
projected days and hours of work based on a 6-week average.  
 
An agency must compute the advance payment based on the projected workdays and work 
hours in the selected time period and on the rate of pay (including any applicable 
allowances, differentials, or other authorized payments) to which the employee was 
regularly entitled immediately before the issuance of the evacuation order.  An employee is 
considered to be regularly entitled to night pay differential and Sunday premium pay for 
applicable hours in the employee’s normal basic workweek.  Also, an employee is 
considered to be regularly entitled to law enforcement availability pay, administratively 
uncontrollable overtime pay, standby duty premium pay, regular overtime pay for 
firefighters covered by 5 U.S.C. 5545b, retention payments under 5 U.S.C. 5754, 
phy

I tested 512 vs 1024 chunk sizes on OPM policy documents and found that 512 characters cut mid-clause on dense government policy text. Since compliance answers require complete rule context to be accurate, I chose 1024 with 128 overlap — accepting higher token cost in exchange for answer reliability

In [11]:
from src.ingestion.vectorstore import VectorStoreManager
from dotenv import load_dotenv
import os
load_dotenv("../.env")  # path relative to notebooks/

api_key = os.getenv("OPENAI_API_KEY")

vm = VectorStoreManager(persist_directory="../data/processed/chroma")
vm.build(chunks_larger)  # using our 1024/128 chunks
print("Vector store built successfully")

Vector store built successfully


In [13]:
retriever = vm.as_retriever()
results = retriever.invoke("What happens to my pay during a weather emergency?")

for doc in results:
    print("---")
    print(f"source: {doc.metadata['source']}")
    print(f"page: {doc.metadata['page']}")
    print(f"content:\n{doc.page_content}")
    print()

---
source: ../data/raw/emergencybenefits.pdf
page: 8
content:
greater number of Federal employees to work and supporting continuity of operations.  
Agencies should continue to promote and incorporate telework into their agency 
emergency planning.  In emergency situations, agencies may continue to designate the 
location of the employee’s reporting office prior to the emergency as the official worksite 
for location-based pay entitlements, such as locality pay and special rates.  (See 5 CFR 
531.605(d)(3).)  Additional information on telework is available at www.telework.gov. 
   
 
II.  Pay and Leave Benefits For Employees Required To Work In an Area Affected   
By Severe Weather Conditions 
or Other Emergency Situations  
 
Work Schedules 
 
Each Federal agency has the authority and responsibility to establish work schedules for its 
employees.  (See 5 U.S.C. 6101.)  Agencies may require employees to perform overtime 
work without any limitation on the number of days or hours.  In 

In [14]:
for chunk in chunks_larger:
    if chunk.metadata.get('page') == 0:
        print(f"length: {len(chunk.page_content)}")
        print(chunk.page_content)

MLflow Experimentation

In [15]:
# import mlflow

# mlflow.set_experiment("compliance-agent-chunking")

# # log the recursive 512/64 experiment
# with mlflow.start_run(run_name="recursive_512_64"):
#     mlflow.log_param("strategy", "recursive")
#     mlflow.log_param("chunk_size", 512)
#     mlflow.log_param("chunk_overlap", 64)
#     mlflow.log_metric("total_chunks", 478)
#     mlflow.log_metric("noise_chunks_filtered", 478 - len(chunks))

# # log the recursive 1024/128 experiment  
# with mlflow.start_run(run_name="recursive_1024_128"):
#     mlflow.log_param("strategy", "recursive")
#     mlflow.log_param("chunk_size", 1024)
#     mlflow.log_param("chunk_overlap", 128)
#     mlflow.log_metric("total_chunks", 267)
#     mlflow.log_metric("noise_chunks_filtered", 267 - len(chunks_larger))

# # log the character experiment
# with mlflow.start_run(run_name="character_512_64"):
#     mlflow.log_param("strategy", "character")
#     mlflow.log_param("chunk_size", 512)
#     mlflow.log_param("chunk_overlap", 64)
#     mlflow.log_metric("total_chunks", 96)

# print("Experiments logged")

In [16]:
import sys
sys.path.append("..")

from src.agent.graph import app
print(type(app))
print("Graph compiled successfully")

<class 'langgraph.graph.state.CompiledStateGraph'>
Graph compiled successfully


In [17]:
from src.agent.nodes.router import route_query

# test factual query
state = {"query": "What happens to my pay during a weather emergency?"}
result = route_query(state)
print(result)

# test conflict query
state = {"query": "How do the elder care and emergency leave policies differ?"}
result = route_query(state)
print(result)

# test ambiguous query
state = {"query": "I'm not sure what to do about my situation"}
result = route_query(state)
print(result)

{'routed_to': 'retriever'}
{'routed_to': 'conflict'}
{'routed_to': 'hitl'}


In [18]:
from src.agent.nodes.retriever import retrieve_documents

state = {"query": "What happens to my pay during a weather emergency?"}
result = retrieve_documents(state)

print(f"Docs retrieved: {len(result['retrieved_docs'])}")
print(f"\nFormatted answer:\n{result['answer']}")

Retriever node found 6 relevant documents.
query: What happens to my pay during a weather emergency?
Docs retrieved: 6

Formatted answer:
During a weather emergency, if your official Federal worksite is closed, your agency may provide weather and safety leave at its discretion. This leave is intended for events like hurricanes, floods, or snowstorms. While on this leave, you will continue to receive your regular pay as if you were working, based on the rates and allowances you were entitled to before the emergency [emergencybenefits.pdf, pages 2-4]. If you are required to evacuate, you may also receive evacuation payments, which will cover the period of evacuation and be based on your regular pay [emergencybenefits.pdf, page 4].


In [19]:
from src.agent.nodes.retriever import retrieve_documents
from src.agent.nodes.critic import critique_answer

# first get retrieved docs and answer
state = {
    "query": "What happens to my pay during a weather emergency?",
    "retry_count": 0,
    "needs_hitl": False
}
state.update(retrieve_documents(state))

# now run the critic
result = critique_answer(state)
print(f"Critique score: {result['critique_score']}")
print(f"Retry count: {result['retry_count']}")

Retriever node found 6 relevant documents.
query: What happens to my pay during a weather emergency?
Critique score: 1.0
Retry count: 1


I found that returning raw retrieved chunks as the answer scored 0.4 on the critic's faithfulness evaluation. Adding an LLM synthesis step that generates a cited answer from the chunks in the retrieval process brought the score to 1.0. The critic loop caught this gap before I would have noticed it manually.

In [20]:
from src.agent.nodes.conflict import detect_conflicts

state = {
    "query": "How do the elder care and emergency leave policies differ on telework?",
    "retry_count": 0,
    "needs_hitl": False
}
result = detect_conflicts(state)
print(result["answer"])

Retriever node found 8 relevant documents.
query: How do the elder care and emergency leave policies differ on telework?
### Analysis of Conflicts Between Policies on Telework

1. **Elder Care Policy on Telework**:
   - The elder care policy allows for situational telework, which is defined as working from an approved location other than the usual worksite for predefined days. This type of telework is to be authorized only when it meets a compelling agency need and should not be used as a substitute for routine telework. Supervisors are expected to exercise discretion in approving such requests, ensuring they align with agency policy and OPM guidance [elder_care_handbook-05062025.pdf, page 21].

2. **Emergency Leave Policy on Telework**:
   - The emergency leave policy states that telework employees are expected to continue working during emergencies and generally cannot receive weather and safety leave. They must account for their entire workday by either teleworking, taking unschedul

Honestly, I am pretty impressed by the accuracy of the comparision.

In [21]:
from langgraph.checkpoint.memory import MemorySaver
from src.agent.graph import build_graph

checkpointer = MemorySaver()
app = build_graph(checkpointer=checkpointer)

# initial state
initial_state = {
    "query": "What happens to my pay during a weather emergency?",
    "retry_count": 0,
    "needs_hitl": False,
    "routed_to": None,
    "retrieved_docs": None,
    "answer": None,
    "critique": None,
    "critique_score": None,
    "hitl_response": None,
    "messages": []
}

# run the graph
config = {"configurable": {"thread_id": "test-1"}}
result = app.invoke(initial_state, config=config)

print(f"Routed to: {result['routed_to']}")
print(f"Critique score: {result['critique_score']}")
print(f"Retry count: {result['retry_count']}")
print(f"\nFinal answer:\n{result['answer']}")

Retriever node found 6 relevant documents.
query: What happens to my pay during a weather emergency?
Routed to: retriever
Critique score: 1.0
Retry count: 1

Final answer:
During a weather emergency, if your official Federal worksite is closed, your agency may provide weather and safety leave at its discretion. This leave does not have specific time limitations and should align with the nature of the emergency. If you are on weather and safety leave, you will continue to receive your regular pay based on the rates and allowances you were entitled to before the emergency [emergencybenefits.pdf, pages 2-4]. 

If you are required to evacuate, your agency may authorize evacuation payments, which will cover the period of evacuation and be based on your regular pay [emergencybenefits.pdf, pages 3-4].


In [22]:
# test 1 - conflict query
conflict_state = {
    "query": "How do the elder care and emergency leave policies differ on telework?",
    "retry_count": 0,
    "needs_hitl": False,
    "routed_to": None,
    "retrieved_docs": None,
    "answer": None,
    "critique": None,
    "critique_score": None,
    "hitl_response": None,
    "messages": []
}
config = {"configurable": {"thread_id": "test-2"}}
result = app.invoke(conflict_state, config=config)
print(f"Routed to: {result['routed_to']}")
print(f"Critique score: {result['critique_score']}")
print(f"\nFinal answer:\n{result['answer']}")

Retriever node found 8 relevant documents.
query: How do the elder care and emergency leave policies differ on telework?
Routed to: conflict
Critique score: 0.9

Final answer:
### Analysis of Conflicts Between Policies on Telework

1. **Elder Care Policy on Telework**:
   - The elder care policy allows for situational telework, which is defined as working from an approved location other than the usual worksite for predefined days. This type of telework is to be authorized only when it meets a compelling agency need and should not be used as a substitute for routine telework. Supervisors are expected to exercise discretion in approving such requests, ensuring they align with agency policy and OPM guidance [elder_care_handbook-05062025.pdf, page 21].

2. **Emergency Leave Policy on Telework**:
   - The emergency leave policy states that telework employees are expected to continue working during emergencies and generally cannot receive weather and safety leave. They must account for their

In [23]:
# test 2 - ambiguous query that should hit HITL
hitl_state = {
    "query": "I have a complicated situation with my leave",
    "retry_count": 0,
    "needs_hitl": False,
    "routed_to": None,
    "retrieved_docs": None,
    "answer": None,
    "critique": None,
    "critique_score": None,
    "hitl_response": None,
    "messages": []
}
config = {"configurable": {"thread_id": "test-3"}}
result = app.invoke(hitl_state, config=config)
print(f"Routed to: {result['routed_to']}")

Routed to: hitl


In [12]:
import mlflow
mlflow.set_tracking_uri("../mlruns")
from src.evals.ragas_eval import run_ragas_eval
results = run_ragas_eval()
print(results)

/Users/monish/Desktop/Monish-Personal/compliance-agent/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Retriever node found 6 relevant documents.
query: What types of natural disasters qualify an employee for weather and safety leave?
Retriever node found 6 relevant documents.
query: What is the maximum number of days used to compute an advance payment during evacuation?
Retriever node found 6 relevant documents.
query: Can an employee care for a family member while teleworking?
Retriever node found 6 relevant documents.
query: How much sick leave can a federal employee use per year to care for a family member with a serious health condition?
Retriever node found 6 relevant documents.
query: Are telework employees eligible for weather and safety leave when the office is closed?
Retriever node found 8 relevant documents.
query: How do the elder care and emergency leave policies differ on telework obligations during emergencies?


Evaluating:   6%|▌         | 1/18 [00:02<00:41,  2.44s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 18/18 [00:36<00:00,  2.02s/it]
/Users/monish/Desktop/Monish-Personal/compliance-agent/venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/10 14:25:57 INFO mlflow.tracking.fluent: Experi

{'faithfulness': 0.7955, 'answer_relevancy': 0.9850, 'context_recall': 0.8889}


In [14]:
from dotenv import load_dotenv
load_dotenv("../.env")

import os
print(os.getenv("PINECONE_API_KEY")[:8])  # verify it loaded

pcsk_5bx


In [15]:
from src.ingestion.loader import PolicyDocumentLoader
from src.ingestion.chunker import PolicyChunker
from src.ingestion.vectorstore import VectorStoreManager
import os
os.environ["PINECONE_API_KEY"] = os.getenv("PINECONE_API_KEY")
os.environ["PINECONE_INDEX_NAME"] = "compliance-agent"

loader = PolicyDocumentLoader("../data/raw")
docs = loader.load()
chunker = PolicyChunker(chunk_size=1024, chunk_overlap=128, strategy="recursive")
chunks = chunker.split(docs)

vm = VectorStoreManager()
vm.build(chunks)
print("Done")

Skipping unsupported file type: ../data/raw/.gitkeep
Done


In [16]:
from src.ingestion.vectorstore import VectorStoreManager

vm = VectorStoreManager()
retriever = vm.as_retriever()
results = retriever.invoke("What happens to my pay during a weather emergency?")

for doc in results:
    print("---")
    print(f"source: {doc.metadata.get('source', '').split('/')[-1]}")
    print(f"page: {doc.metadata.get('page')}")
    print(f"content preview: {doc.page_content[:150]}")

---
source: emergencybenefits.pdf
page: 8
content preview: greater number of Federal employees to work and supporting continuity of operations.  
Agencies should continue to promote and incorporate telework in
---
source: emergencybenefits.pdf
page: 2
content preview: 1 
I. Pay and Leave Benefits For Employees Prevented From Working In an Area
Affected By Severe Weather Conditions or Other Emergency Situations
Weath
---
source: emergencybenefits.pdf
page: 4
content preview: employee’s pay on the basis of the rates of pay, allowances, and differentials, if any, to 
which the employee otherwise would have been entitled unde
---
source: emergencybenefits.pdf
page: 3
content preview: biweekly pay period).  If possible, the agency should estimate an intermittent employee’s 
projected days and hours of work based on a 6-week average.
---
source: emergencybenefits.pdf
page: 4
content preview: allowances, differentials, or other authorized payments) to which the employee was 
regularly entitle

In [17]:
from langgraph.checkpoint.memory import MemorySaver
from src.agent.graph import build_graph

checkpointer = MemorySaver()
app = build_graph(checkpointer=checkpointer)

result = app.invoke({
    "query": "What happens to my pay during a weather emergency?",
    "retry_count": 0,
    "needs_hitl": False,
    "routed_to": None,
    "retrieved_docs": None,
    "answer": None,
    "critique": None,
    "critique_score": None,
    "hitl_response": None,
    "messages": []
}, config={"configurable": {"thread_id": "pinecone-test-1"}})

print(f"Routed to: {result['routed_to']}")
print(f"Critique score: {result['critique_score']}")
print(f"\nAnswer:\n{result['answer']}")

Retriever node found 6 relevant documents.
query: What happens to my pay during a weather emergency?
Routed to: retriever
Critique score: 1.0

Answer:
During a weather emergency, if your official Federal worksite is closed, your agency may provide weather and safety leave at its discretion. This leave is intended for events like hurricanes, floods, or snowstorms. While on this leave, you will continue to receive your regular pay based on the rates and allowances you were entitled to before the emergency [emergencybenefits.pdf, page 2; page 4]. If you are required to evacuate, you may also receive evacuation payments, which will cover the period of evacuation and be based on your regular pay [emergencybenefits.pdf, page 4].


In [ ]:
import sys
sys.path.append("..")  # so we can import from src/

from src.ingestion.loader import PolicyDocumentLoader
from src.ingestion.chunker import PolicyChunker

In [3]:
from dotenv import load_dotenv
load_dotenv("../.env")

import os
from pinecone import Pinecone

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("compliance-agent")
index.delete(delete_all=True)
print("Index cleared")
print(index.describe_index_stats())

/Users/monish/Desktop/Monish-Personal/compliance-agent/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NotFoundException: (404)
Reason: Not Found
HTTP response headers: HTTPHeaderDict({'Date': 'Sat, 11 Apr 2026 23:37:58 GMT', 'Content-Type': 'application/json', 'Content-Length': '55', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '131', 'x-envoy-upstream-service-time': '126', 'x-pinecone-response-duration-ms': '132', 'server': 'envoy'})
HTTP response body: {"code":5,"message":"Namespace not found","details":[]}


In [4]:
from src.ingestion.loader import PolicyDocumentLoader
from src.ingestion.chunker import PolicyChunker
from src.ingestion.vectorstore import VectorStoreManager

loader = PolicyDocumentLoader("../data/raw")
docs = loader.load()

chunker = PolicyChunker(chunk_size=1024, chunk_overlap=128, strategy="recursive")
chunks = chunker.split(docs)

print(f"Chunks: {len(chunks)}")
print(f"Sample page type: {type(chunks[0].metadata['page'])}")

vm = VectorStoreManager()
vm.build(chunks)
print("Done")

Skipping unsupported file type: ../data/raw/.gitkeep
Chunks: 250
Sample page type: <class 'int'>
Upserted 250 chunks to Pinecone index 'compliance-agent'
Done
